# ESG-Driven Portfolio Optimization - Data Pipeline
## Big Data Analytics (EECS-6893) Project
### Data Extraction, Cleaning, Preprocessing & Validation

This notebook extracts stock price data and ESG metrics, cleans and preprocesses them, and validates that the data is suitable for multi-objective portfolio optimization using Reinforcement Learning and Pareto Frontier models.

## 1. Install Required Libraries

In [1]:
!pip install yfinance pandas numpy scikit-learn matplotlib seaborn ta scipy

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=a2c3a40e9c1e3db36f929059032d641dcef953ae4c12bce3a51df4e98ef11c98
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


## 2. Import Libraries

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 3. Define Stock Universe
We'll use a diverse set of stocks across multiple sectors to capture varying ESG profiles.

In [3]:
# Define tickers across sectors for ESG diversity
TICKERS = [
    # Technology (typically high ESG)
    'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'CRM',
    # Healthcare
    'JNJ', 'PFE', 'UNH', 'MRK', 'ABBV',
    # Financials
    'JPM', 'BAC', 'GS', 'BLK', 'MS',
    # Energy (typically lower ESG)
    'XOM', 'CVX', 'COP', 'SLB', 'EOG',
    # Utilities (mixed ESG)
    'NEE', 'DUK', 'SO', 'D', 'AEP',
    # Consumer Staples
    'PG', 'KO', 'PEP', 'WMT', 'COST',
    # Clean Energy / EV (high ESG)
    'TSLA', 'ENPH', 'FSLR', 'PLUG', 'RUN',
    # Industrials
    'CAT', 'DE', 'HON', 'UPS', 'RTX'
]

SECTORS = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'GOOGL': 'Technology',
    'NVDA': 'Technology', 'CRM': 'Technology',
    'JNJ': 'Healthcare', 'PFE': 'Healthcare', 'UNH': 'Healthcare',
    'MRK': 'Healthcare', 'ABBV': 'Healthcare',
    'JPM': 'Financials', 'BAC': 'Financials', 'GS': 'Financials',
    'BLK': 'Financials', 'MS': 'Financials',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy',
    'SLB': 'Energy', 'EOG': 'Energy',
    'NEE': 'Utilities', 'DUK': 'Utilities', 'SO': 'Utilities',
    'D': 'Utilities', 'AEP': 'Utilities',
    'PG': 'Consumer Staples', 'KO': 'Consumer Staples', 'PEP': 'Consumer Staples',
    'WMT': 'Consumer Staples', 'COST': 'Consumer Staples',
    'TSLA': 'Clean Energy', 'ENPH': 'Clean Energy', 'FSLR': 'Clean Energy',
    'PLUG': 'Clean Energy', 'RUN': 'Clean Energy',
    'CAT': 'Industrials', 'DE': 'Industrials', 'HON': 'Industrials',
    'UPS': 'Industrials', 'RTX': 'Industrials'
}

print(f"📊 Stock Universe: {len(TICKERS)} tickers across {len(set(SECTORS.values()))} sectors")
for sector in sorted(set(SECTORS.values())):
    tickers_in_sector = [t for t, s in SECTORS.items() if s == sector]
    print(f"   {sector}: {', '.join(tickers_in_sector)}")

📊 Stock Universe: 40 tickers across 8 sectors
   Clean Energy: TSLA, ENPH, FSLR, PLUG, RUN
   Consumer Staples: PG, KO, PEP, WMT, COST
   Energy: XOM, CVX, COP, SLB, EOG
   Financials: JPM, BAC, GS, BLK, MS
   Healthcare: JNJ, PFE, UNH, MRK, ABBV
   Industrials: CAT, DE, HON, UPS, RTX
   Technology: AAPL, MSFT, GOOGL, NVDA, CRM
   Utilities: NEE, DUK, SO, D, AEP


## 4. Extract Stock Price Data (Yahoo Finance)
This demonstrates the **Velocity** (daily updates) and **Volume** (thousands of records) aspects of Big Data.

In [4]:
# Download 3 years of historical data
END_DATE = datetime.now().strftime('%Y-%m-%d')
START_DATE = (datetime.now() - timedelta(days=3*365)).strftime('%Y-%m-%d')

print(f"📥 Downloading stock data from {START_DATE} to {END_DATE}...")
print("This may take a minute...\n")

# Download all tickers at once for efficiency
stock_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    group_by='ticker',
    auto_adjust=True,
    progress=True
)

print(f"\n✅ Downloaded data shape: {stock_data.shape}")

📥 Downloading stock data from 2022-11-16 to 2025-11-15...
This may take a minute...



[*********************100%***********************]  40 of 40 completed


✅ Downloaded data shape: (752, 200)


In [5]:
# Reshape data into long format for easier processing
price_records = []

for ticker in TICKERS:
    try:
        ticker_data = stock_data[ticker].copy()
        ticker_data['Ticker'] = ticker
        ticker_data['Sector'] = SECTORS[ticker]
        ticker_data = ticker_data.reset_index()
        price_records.append(ticker_data)
    except Exception as e:
        print(f"⚠️ Error processing {ticker}: {e}")

price_df = pd.concat(price_records, ignore_index=True)
price_df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Sector']

print(f"📊 Total price records: {len(price_df):,}")
print(f"   Date range: {price_df['Date'].min().strftime('%Y-%m-%d')} to {price_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"   Unique tickers: {price_df['Ticker'].nunique()}")
print(f"\nFirst few records:")
price_df.head()

📊 Total price records: 30,080
   Date range: 2022-11-16 to 2025-11-14
   Unique tickers: 40

First few records:


,Date,Open,High,Low,Close,Volume,Ticker,Sector
0,2022-11-16,146.9137,147.6427,145.1010,146.5787,64218300,AAPL,Technology
1,2022-11-17,144.2537,149.2287,143.9779,148.4800,80389400,AAPL,Technology
2,2022-11-18,150.0464,150.4306,147.7411,149.0415,74829600,AAPL,Technology
3,2022-11-21,147.9283,148.1352,145.5246,145.8103,58724100,AAPL,Technology
4,2022-11-22,145.9285,148.1845,144.7463,147.9480,51804100,AAPL,Technology


## 5. Extract ESG Data
We'll use Yahoo Finance's sustainability data and supplement with calculated metrics.

In [6]:
print("📥 Extracting ESG/Sustainability data from Yahoo Finance...\n")

esg_records = []

for ticker in TICKERS:
    try:
        stock = yf.Ticker(ticker)

        # Get sustainability data
        sustainability = stock.sustainability

        if sustainability is not None and not sustainability.empty:
            # Extract ESG scores
            esg_data = {
                'Ticker': ticker,
                'Sector': SECTORS[ticker]
            }

            # Map common ESG metrics
            metric_mapping = {
                'totalEsg': 'Total_ESG_Score',
                'environmentScore': 'E_Score',
                'socialScore': 'S_Score',
                'governanceScore': 'G_Score',
                'esgPerformance': 'ESG_Performance',
                'percentile': 'ESG_Percentile',
                'peerCount': 'Peer_Count',
                'peerGroup': 'Peer_Group',
                'highestControversy': 'Controversy_Level'
            }

            for key, col_name in metric_mapping.items():
                if key in sustainability.index:
                    esg_data[col_name] = sustainability.loc[key].values[0]
                else:
                    esg_data[col_name] = np.nan

            esg_records.append(esg_data)
            print(f"✅ {ticker}: ESG Score = {esg_data.get('Total_ESG_Score', 'N/A')}")
        else:
            # Create placeholder with NaN for missing data
            esg_records.append({
                'Ticker': ticker,
                'Sector': SECTORS[ticker],
                'Total_ESG_Score': np.nan,
                'E_Score': np.nan,
                'S_Score': np.nan,
                'G_Score': np.nan
            })
            print(f"⚠️ {ticker}: No ESG data available")

    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)[:50]}")
        esg_records.append({
            'Ticker': ticker,
            'Sector': SECTORS[ticker],
            'Total_ESG_Score': np.nan
        })

esg_df = pd.DataFrame(esg_records)
print(f"\n📊 ESG data extracted for {len(esg_df)} companies")

📥 Extracting ESG/Sustainability data from Yahoo Finance...



ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: AAPL"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: MSFT"}}}


⚠️ AAPL: No ESG data available
⚠️ MSFT: No ESG data available


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: GOOGL"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: NVDA"}}}


⚠️ GOOGL: No ESG data available
⚠️ NVDA: No ESG data available


...

⚠️ DE: No ESG data available
⚠️ HON: No ESG data available


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: UPS"}}}


⚠️ UPS: No ESG data available


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: RTX"}}}


⚠️ RTX: No ESG data available

📊 ESG data extracted for 40 companies


In [7]:
# Display ESG data
print("ESG Scores Summary:")
esg_df.head(10)

ESG Scores Summary:


,Ticker,Sector,Total_ESG_Score,E_Score,S_Score,G_Score
0,AAPL,Technology,NaN,NaN,NaN,NaN
1,MSFT,Technology,NaN,NaN,NaN,NaN
2,GOOGL,Technology,NaN,NaN,NaN,NaN
3,NVDA,Technology,NaN,NaN,NaN,NaN
4,CRM,Technology,NaN,NaN,NaN,NaN
5,JNJ,Healthcare,NaN,NaN,NaN,NaN
6,PFE,Healthcare,NaN,NaN,NaN,NaN
7,UNH,Healthcare,NaN,NaN,NaN,NaN
8,MRK,Healthcare,NaN,NaN,NaN,NaN
9,ABBV,Healthcare,NaN,NaN,NaN,NaN


## 6. Data Cleaning
Handle missing values, outliers, and data quality issues.

In [8]:
print("🧹 CLEANING STOCK PRICE DATA\n")
print("=" * 50)

# Check for missing values
print("Missing values in price data:")
missing_prices = price_df.isnull().sum()
print(missing_prices[missing_prices > 0])

# Check for duplicates
duplicates = price_df.duplicated(subset=['Date', 'Ticker']).sum()
print(f"\nDuplicate records: {duplicates}")

# Remove duplicates if any
price_df_clean = price_df.drop_duplicates(subset=['Date', 'Ticker'])

# Handle missing prices using forward fill within each ticker
price_df_clean = price_df_clean.sort_values(['Ticker', 'Date'])
price_df_clean = price_df_clean.groupby('Ticker').apply(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
).reset_index(drop=True)

# Check for outliers (prices that changed more than 50% in a day)
price_df_clean['Daily_Return'] = price_df_clean.groupby('Ticker')['Close'].pct_change()
outliers = price_df_clean[abs(price_df_clean['Daily_Return']) > 0.5]
print(f"\nPotential outliers (>50% daily change): {len(outliers)}")

# Validate price data integrity
invalid_prices = price_df_clean[
    (price_df_clean['High'] < price_df_clean['Low']) |
    (price_df_clean['Close'] < 0) |
    (price_df_clean['Volume'] < 0)
]
print(f"Invalid price records: {len(invalid_prices)}")

print(f"\n✅ Cleaned price data: {len(price_df_clean):,} records")

🧹 CLEANING STOCK PRICE DATA

Missing values in price data:
Series([], dtype: int64)

Duplicate records: 0

Potential outliers (>50% daily change): 0
Invalid price records: 0

✅ Cleaned price data: 30,080 records


In [9]:
print("🧹 CLEANING ESG DATA\n")
print("=" * 50)

# Check missing ESG values
print("Missing values in ESG data:")
esg_missing = esg_df.isnull().sum()
print(esg_missing[esg_missing > 0])

# For missing ESG scores, impute using sector median
esg_df_clean = esg_df.copy()

# Impute missing ESG scores by sector median
score_columns = ['Total_ESG_Score', 'E_Score', 'S_Score', 'G_Score']

for col in score_columns:
    if col in esg_df_clean.columns:
        # First try sector median
        esg_df_clean[col] = esg_df_clean.groupby('Sector')[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Then fill any remaining with overall median
        esg_df_clean[col] = esg_df_clean[col].fillna(esg_df_clean[col].median())

print(f"\nAfter imputation - Missing values:")
print(esg_df_clean[score_columns].isnull().sum())

# Validate ESG score ranges (should be 0-100)
for col in score_columns:
    if col in esg_df_clean.columns:
        out_of_range = esg_df_clean[
            (esg_df_clean[col] < 0) | (esg_df_clean[col] > 100)
        ]
        if len(out_of_range) > 0:
            print(f"\n⚠️ {col} out of range (0-100): {len(out_of_range)} records")
            # Clip to valid range
            esg_df_clean[col] = esg_df_clean[col].clip(0, 100)

print(f"\n✅ Cleaned ESG data: {len(esg_df_clean)} companies")

🧹 CLEANING ESG DATA

Missing values in ESG data:
Total_ESG_Score    40
E_Score            40
S_Score            40
G_Score            40
dtype: int64

After imputation - Missing values:
Total_ESG_Score    40
E_Score            40
S_Score            40
G_Score            40
dtype: int64

✅ Cleaned ESG data: 40 companies


## 7. Feature Engineering
Create features for portfolio optimization and RL models.

In [10]:
print("⚙️ FEATURE ENGINEERING - FINANCIAL METRICS\n")
print("=" * 50)

# Calculate key financial features for each stock
financial_features = []

for ticker in TICKERS:
    ticker_data = price_df_clean[price_df_clean['Ticker'] == ticker].sort_values('Date').copy()

    if len(ticker_data) < 252:  # Need at least 1 year of data
        continue

    # Calculate returns
    ticker_data['Daily_Return'] = ticker_data['Close'].pct_change()
    ticker_data['Log_Return'] = np.log(ticker_data['Close'] / ticker_data['Close'].shift(1))

    # Calculate key metrics
    features = {
        'Ticker': ticker,
        'Sector': SECTORS[ticker],

        # Returns
        'Mean_Daily_Return': ticker_data['Daily_Return'].mean(),
        'Annualized_Return': ticker_data['Daily_Return'].mean() * 252,

        # Risk metrics
        'Daily_Volatility': ticker_data['Daily_Return'].std(),
        'Annualized_Volatility': ticker_data['Daily_Return'].std() * np.sqrt(252),

        # Risk-adjusted metrics
        'Sharpe_Ratio': (ticker_data['Daily_Return'].mean() * 252) /
                        (ticker_data['Daily_Return'].std() * np.sqrt(252)) if ticker_data['Daily_Return'].std() > 0 else 0,

        # Downside risk
        'Max_Drawdown': (ticker_data['Close'] / ticker_data['Close'].cummax() - 1).min(),
        'Downside_Deviation': ticker_data[ticker_data['Daily_Return'] < 0]['Daily_Return'].std(),

        # Skewness and Kurtosis
        'Skewness': ticker_data['Daily_Return'].skew(),
        'Kurtosis': ticker_data['Daily_Return'].kurtosis(),

        # Trading metrics
        'Avg_Volume': ticker_data['Volume'].mean(),
        'Volume_Volatility': ticker_data['Volume'].std() / ticker_data['Volume'].mean(),

        # Price levels
        'Current_Price': ticker_data['Close'].iloc[-1],
        'Price_52W_High': ticker_data['Close'].tail(252).max(),
        'Price_52W_Low': ticker_data['Close'].tail(252).min(),

        # Momentum indicators
        'Return_1M': (ticker_data['Close'].iloc[-1] / ticker_data['Close'].iloc[-21] - 1) if len(ticker_data) > 21 else np.nan,
        'Return_3M': (ticker_data['Close'].iloc[-1] / ticker_data['Close'].iloc[-63] - 1) if len(ticker_data) > 63 else np.nan,
        'Return_6M': (ticker_data['Close'].iloc[-1] / ticker_data['Close'].iloc[-126] - 1) if len(ticker_data) > 126 else np.nan,
        'Return_1Y': (ticker_data['Close'].iloc[-1] / ticker_data['Close'].iloc[-252] - 1) if len(ticker_data) > 252 else np.nan,
    }

    financial_features.append(features)

financial_df = pd.DataFrame(financial_features)
print(f"✅ Calculated {len(financial_df.columns) - 2} financial features for {len(financial_df)} stocks")
print(f"\nKey Features:")
print(financial_df[['Ticker', 'Annualized_Return', 'Annualized_Volatility', 'Sharpe_Ratio', 'Max_Drawdown']].head(10))

⚙️ FEATURE ENGINEERING - FINANCIAL METRICS

✅ Calculated 18 financial features for 40 stocks

Key Features:
  Ticker  Annualized_Return  Annualized_Volatility  Sharpe_Ratio  Max_Drawdown
0   AAPL             0.2425                 0.2635        0.9201       -0.3336
1   MSFT             0.2857                 0.2361        1.2104       -0.2373
2  GOOGL             0.3935                 0.3033        1.2970       -0.2981
3   NVDA             0.9623                 0.5103        1.8857       -0.3688
4    CRM             0.2113                 0.3335        0.6336       -0.3676
5    JNJ             0.0860                 0.1700        0.5056       -0.1743
6    PFE            -0.1309                 0.2421       -0.5408       -0.5531
7    UNH            -0.0790                 0.3377       -0.2339       -0.6139
8    MRK             0.0332                 0.2341        0.1417       -0.4344
9   ABBV             0.2065                 0.2307        0.8954       -0.2074


In [11]:
# Merge financial and ESG data
print("\n🔗 MERGING FINANCIAL AND ESG DATA\n")
print("=" * 50)

combined_df = financial_df.merge(esg_df_clean, on=['Ticker', 'Sector'], how='left')

print(f"Combined dataset shape: {combined_df.shape}")
print(f"Features: {list(combined_df.columns)}")

# Create composite ESG-adjusted metrics
if 'Total_ESG_Score' in combined_df.columns:
    # ESG-adjusted Sharpe Ratio
    combined_df['ESG_Adjusted_Sharpe'] = combined_df['Sharpe_Ratio'] * (combined_df['Total_ESG_Score'] / 50)

    # ESG Penalty Factor (lower ESG = higher penalty)
    combined_df['ESG_Penalty'] = 1 - (combined_df['Total_ESG_Score'] / 100)

    # Risk-Return-ESG Score
    combined_df['RRE_Score'] = (
        0.4 * combined_df['Annualized_Return'] / combined_df['Annualized_Return'].max() +
        0.3 * (1 - combined_df['Annualized_Volatility'] / combined_df['Annualized_Volatility'].max()) +
        0.3 * combined_df['Total_ESG_Score'] / 100
    )

print("\n✅ Added ESG-adjusted metrics:")
print("   - ESG_Adjusted_Sharpe")
print("   - ESG_Penalty")
print("   - RRE_Score (Risk-Return-ESG)")


🔗 MERGING FINANCIAL AND ESG DATA

Combined dataset shape: (40, 24)
Features: ['Ticker', 'Sector', 'Mean_Daily_Return', 'Annualized_Return', 'Daily_Volatility', 'Annualized_Volatility', 'Sharpe_Ratio', 'Max_Drawdown', 'Downside_Deviation', 'Skewness', 'Kurtosis', 'Avg_Volume', 'Volume_Volatility', 'Current_Price', 'Price_52W_High', 'Price_52W_Low', 'Return_1M', 'Return_3M', 'Return_6M', 'Return_1Y', 'Total_ESG_Score', 'E_Score', 'S_Score', 'G_Score']

✅ Added ESG-adjusted metrics:
   - ESG_Adjusted_Sharpe
   - ESG_Penalty
   - RRE_Score (Risk-Return-ESG)


In [13]:
print(combined_df.head())

  Ticker      Sector  Mean_Daily_Return  Annualized_Return  Daily_Volatility  \
0   AAPL  Technology             0.0010             0.2425            0.0166   
1   MSFT  Technology             0.0011             0.2857            0.0149   
2  GOOGL  Technology             0.0016             0.3935            0.0191   
3   NVDA  Technology             0.0038             0.9623            0.0321   
4    CRM  Technology             0.0008             0.2113            0.0210   

   Annualized_Volatility  Sharpe_Ratio  Max_Drawdown  Downside_Deviation  \
0                 0.2635        0.9201       -0.3336              0.0116   
1                 0.2361        1.2104       -0.2373              0.0096   
2                 0.3033        1.2970       -0.2981              0.0130   
3                 0.5103        1.8857       -0.3688              0.0200   
4                 0.3335        0.6336       -0.3676              0.0160   

   Skewness  Kurtosis     Avg_Volume  Volume_Volatility  Curre

## 8. Data Preprocessing for ML/RL Models

In [12]:
print("🔧 PREPROCESSING FOR OPTIMIZATION MODELS\n")
print("=" * 50)

# Select features for modeling
feature_columns = [
    'Annualized_Return', 'Annualized_Volatility', 'Sharpe_Ratio',
    'Max_Drawdown', 'Downside_Deviation', 'Skewness', 'Kurtosis',
    'Return_1M', 'Return_3M', 'Return_6M', 'Return_1Y'
]

# Add ESG features if available
esg_feature_cols = ['Total_ESG_Score', 'E_Score', 'S_Score', 'G_Score']
for col in esg_feature_cols:
    if col in combined_df.columns:
        feature_columns.append(col)

# Extract feature matrix
X = combined_df[feature_columns].copy()

# Handle any remaining missing values
print(f"Missing values before imputation:")
print(X.isnull().sum()[X.isnull().sum() > 0])

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_columns)

print(f"\nMissing values after imputation: {X_imputed.isnull().sum().sum()}")

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=feature_columns
)

print(f"\n✅ Preprocessed feature matrix shape: {X_scaled.shape}")
print(f"\nFeature statistics after scaling:")
X_scaled.describe().round(4)

🔧 PREPROCESSING FOR OPTIMIZATION MODELS

Missing values before imputation:
Total_ESG_Score    40
E_Score            40
S_Score            40
G_Score            40
dtype: int64


ValueError: Shape of passed values is (40, 11), indices imply (40, 15)

In [ ]:
# Create correlation matrix to check for multicollinearity
print("📊 FEATURE CORRELATION ANALYSIS\n")

plt.figure(figsize=(14, 10))
corr_matrix = X_imputed.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Identify highly correlated features
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("⚠️ Highly correlated feature pairs (|r| > 0.8):")
    for pair in high_corr_pairs:
        print(f"   {pair[0]} <-> {pair[1]}: {pair[2]:.3f}")
else:
    print("✅ No highly correlated feature pairs found")

## 9. Validation for Portfolio Optimization

In [ ]:
print("✅ DATA VALIDATION FOR ESG PORTFOLIO OPTIMIZATION\n")
print("=" * 60)

validation_results = {}

# 1. Check data completeness
print("1️⃣ DATA COMPLETENESS")
completeness_score = 1 - (combined_df.isnull().sum().sum() / (combined_df.shape[0] * combined_df.shape[1]))
validation_results['completeness'] = completeness_score
print(f"   Overall completeness: {completeness_score:.2%}")
print(f"   ✅ PASS" if completeness_score > 0.9 else "   ⚠️ WARNING: Low completeness")

# 2. Check for sufficient historical data
print("\n2️⃣ HISTORICAL DATA SUFFICIENCY")
records_per_ticker = price_df_clean.groupby('Ticker').size()
min_records = records_per_ticker.min()
validation_results['min_history'] = min_records
print(f"   Minimum trading days per stock: {min_records}")
print(f"   Required for 1-year analysis: 252 days")
print(f"   ✅ PASS" if min_records >= 252 else "   ⚠️ WARNING: Insufficient history")

# 3. Check ESG score distribution
print("\n3️⃣ ESG SCORE DISTRIBUTION")
if 'Total_ESG_Score' in combined_df.columns:
    esg_range = combined_df['Total_ESG_Score'].max() - combined_df['Total_ESG_Score'].min()
    esg_std = combined_df['Total_ESG_Score'].std()
    validation_results['esg_range'] = esg_range
    validation_results['esg_std'] = esg_std
    print(f"   ESG Score Range: {esg_range:.2f}")
    print(f"   ESG Score Std Dev: {esg_std:.2f}")
    print(f"   ✅ PASS" if esg_range > 20 else "   ⚠️ WARNING: Limited ESG diversity")

# 4. Check sector diversity
print("\n4️⃣ SECTOR DIVERSITY")
n_sectors = combined_df['Sector'].nunique()
validation_results['n_sectors'] = n_sectors
print(f"   Number of sectors: {n_sectors}")
print(f"   Sector distribution:")
print(combined_df['Sector'].value_counts().to_string())
print(f"   ✅ PASS" if n_sectors >= 5 else "   ⚠️ WARNING: Low sector diversity")

# 5. Check for Pareto-optimality feasibility
print("\n5️⃣ PARETO FRONTIER FEASIBILITY")
if 'Total_ESG_Score' in combined_df.columns:
    # Check correlation between returns and ESG (should not be too high)
    corr_ret_esg = combined_df['Annualized_Return'].corr(combined_df['Total_ESG_Score'])
    validation_results['return_esg_corr'] = corr_ret_esg
    print(f"   Correlation (Return vs ESG): {corr_ret_esg:.3f}")
    print(f"   Interpretation: {'Low' if abs(corr_ret_esg) < 0.3 else 'Moderate' if abs(corr_ret_esg) < 0.6 else 'High'} correlation")
    print(f"   ✅ PASS - Trade-off exists" if abs(corr_ret_esg) < 0.7 else "   ⚠️ High correlation - may limit Pareto frontier")

# 6. Check for RL suitability
print("\n6️⃣ REINFORCEMENT LEARNING SUITABILITY")
n_features = len(feature_columns)
n_samples = len(combined_df)
validation_results['n_features'] = n_features
validation_results['n_samples'] = n_samples
print(f"   Number of features: {n_features}")
print(f"   Number of assets: {n_samples}")
print(f"   State space dimensionality: {n_features} continuous features")
print(f"   Action space: Portfolio weights (continuous, {n_samples}-dimensional)")
print(f"   ✅ PASS - Suitable for RL" if n_samples >= 20 and n_features >= 10 else "   ⚠️ May need more data")

print("\n" + "=" * 60)
print("📋 VALIDATION SUMMARY")
print("=" * 60)

all_pass = True
checks = [
    ("Data Completeness", validation_results.get('completeness', 0) > 0.9),
    ("Historical Sufficiency", validation_results.get('min_history', 0) >= 252),
    ("ESG Diversity", validation_results.get('esg_range', 0) > 20),
    ("Sector Diversity", validation_results.get('n_sectors', 0) >= 5),
    ("Pareto Feasibility", abs(validation_results.get('return_esg_corr', 1)) < 0.7),
    ("RL Suitability", validation_results.get('n_samples', 0) >= 20)
]

for check_name, passed in checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"   {check_name}: {status}")
    all_pass = all_pass and passed

print("\n" + "=" * 60)
if all_pass:
    print("🎉 ALL VALIDATIONS PASSED!")
    print("The data is suitable for ESG-driven portfolio optimization using:")
    print("   • Multi-objective optimization (Pareto Frontier)")
    print("   • Reinforcement Learning for dynamic allocation")
    print("   • ESG penalty/reward functions")
    print("   • Sharpe ratio maximization with ESG constraints")
else:
    print("⚠️ SOME VALIDATIONS FAILED - Review warnings above")
print("=" * 60)

## 10. Visualize Data for Insights

In [ ]:
# ESG vs Returns Scatter Plot (Pareto Frontier Preview)
if 'Total_ESG_Score' in combined_df.columns:
    plt.figure(figsize=(12, 8))

    # Create scatter plot colored by sector
    sectors = combined_df['Sector'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(sectors)))

    for idx, sector in enumerate(sectors):
        sector_data = combined_df[combined_df['Sector'] == sector]
        plt.scatter(sector_data['Total_ESG_Score'],
                   sector_data['Annualized_Return'] * 100,
                   c=[colors[idx]], label=sector, s=100, alpha=0.7, edgecolor='black')

        # Add ticker labels
        for _, row in sector_data.iterrows():
            plt.annotate(row['Ticker'],
                        (row['Total_ESG_Score'], row['Annualized_Return'] * 100),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)

    plt.xlabel('ESG Score', fontsize=12)
    plt.ylabel('Annualized Return (%)', fontsize=12)
    plt.title('ESG Score vs Annualized Return by Sector\n(Pareto Frontier Preview)', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\n📊 This visualization shows the trade-off space for your multi-objective optimization.")
    print("   The Pareto frontier will identify stocks that offer optimal combinations of")
    print("   high returns AND high ESG scores.")

In [ ]:
# Risk-Return-ESG 3D visualization
if 'Total_ESG_Score' in combined_df.columns:
    from mpl_toolkits.mplot3d import Axes3D

    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(
        combined_df['Annualized_Volatility'] * 100,
        combined_df['Annualized_Return'] * 100,
        combined_df['Total_ESG_Score'],
        c=combined_df['Sharpe_Ratio'],
        cmap='viridis',
        s=100,
        alpha=0.7,
        edgecolor='black'
    )

    ax.set_xlabel('Annualized Volatility (%)', fontsize=11)
    ax.set_ylabel('Annualized Return (%)', fontsize=11)
    ax.set_zlabel('ESG Score', fontsize=11)
    ax.set_title('3D Risk-Return-ESG Space\n(Color = Sharpe Ratio)', fontsize=14)

    plt.colorbar(scatter, ax=ax, label='Sharpe Ratio', shrink=0.6)
    plt.tight_layout()
    plt.show()

    print("\n📊 This 3D visualization represents the multi-objective optimization space:")
    print("   • X-axis: Risk (volatility)")
    print("   • Y-axis: Return (annualized)")
    print("   • Z-axis: ESG Score")
    print("   • Color: Sharpe Ratio (risk-adjusted return)")

In [ ]:
# ESG Score distribution by sector
if 'Total_ESG_Score' in combined_df.columns:
    plt.figure(figsize=(12, 6))

    sector_order = combined_df.groupby('Sector')['Total_ESG_Score'].mean().sort_values(ascending=False).index

    sns.boxplot(data=combined_df, x='Sector', y='Total_ESG_Score', order=sector_order)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Sector', fontsize=12)
    plt.ylabel('ESG Score', fontsize=12)
    plt.title('ESG Score Distribution by Sector', fontsize=14)
    plt.tight_layout()
    plt.show()

## 11. Save Processed Data

In [ ]:
print("💾 SAVING PROCESSED DATA\n")
print("=" * 50)

# Save price data
price_df_clean.to_csv('stock_prices_cleaned.csv', index=False)
print(f"✅ Stock prices saved: stock_prices_cleaned.csv ({len(price_df_clean):,} records)")

# Save ESG data
esg_df_clean.to_csv('esg_scores_cleaned.csv', index=False)
print(f"✅ ESG scores saved: esg_scores_cleaned.csv ({len(esg_df_clean)} companies)")

# Save combined features
combined_df.to_csv('combined_features.csv', index=False)
print(f"✅ Combined features saved: combined_features.csv ({len(combined_df)} stocks, {len(combined_df.columns)} features)")

# Save scaled features for ML
X_scaled['Ticker'] = combined_df['Ticker'].values
X_scaled.to_csv('features_scaled.csv', index=False)
print(f"✅ Scaled features saved: features_scaled.csv")

# Save validation results
pd.DataFrame([validation_results]).to_csv('validation_results.csv', index=False)
print(f"✅ Validation results saved: validation_results.csv")

print("\n📁 All files saved successfully!")
print("   These files are ready for:")
print("   • Reinforcement Learning model training")
print("   • Pareto Frontier optimization")
print("   • ESG constraint optimization")
print("   • Dashboard visualization")

## 12. Summary Statistics

In [ ]:
print("="*70)
print("📊 FINAL DATA PIPELINE SUMMARY")
print("="*70)

print(f"\n📈 STOCK PRICE DATA (Velocity + Volume)")
print(f"   Total records: {len(price_df_clean):,}")
print(f"   Date range: {price_df_clean['Date'].min().strftime('%Y-%m-%d')} to {price_df_clean['Date'].max().strftime('%Y-%m-%d')}")
print(f"   Number of tickers: {price_df_clean['Ticker'].nunique()}")
print(f"   Trading days per stock: ~{len(price_df_clean) // price_df_clean['Ticker'].nunique():,}")

print(f"\n🌱 ESG DATA (Variety)")
print(f"   Companies with ESG scores: {esg_df_clean['Total_ESG_Score'].notna().sum()}")
if 'Total_ESG_Score' in combined_df.columns:
    print(f"   ESG Score range: {combined_df['Total_ESG_Score'].min():.2f} - {combined_df['Total_ESG_Score'].max():.2f}")
    print(f"   Mean ESG Score: {combined_df['Total_ESG_Score'].mean():.2f}")

print(f"\n🎯 COMBINED FEATURE SET")
print(f"   Total features: {len(combined_df.columns)}")
print(f"   Financial features: {len([c for c in combined_df.columns if 'Return' in c or 'Volatility' in c or 'Sharpe' in c])}")
print(f"   ESG features: {len([c for c in combined_df.columns if 'ESG' in c or 'Score' in c])}")
print(f"   Sectors represented: {combined_df['Sector'].nunique()}")

print(f"\n✅ DATA READY FOR:")
print(f"   • Multi-objective optimization (maximize returns + ESG)")
print(f"   • Reinforcement Learning (state = features, action = weights)")
print(f"   • Pareto Frontier analysis")
print(f"   • ESG penalty/reward function design")
print(f"   • Constraint optimization (min ESG threshold)")
print(f"   • Sharpe ratio maximization with ESG integration")

print("\n" + "="*70)
print("🚀 DATA PIPELINE COMPLETE - Ready for Model Development!")
print("="*70)